Import the chosen model

In [ ]:
import sys
import os
import pandas as pd

# Add the absolute path to the scripts folder
scripts_path = os.path.abspath("../scripts")
sys.path.append(scripts_path)

from utils import load_model
from evaluation import analyze_all_categorical_features


In [32]:
test_final = pd.read_csv("../data/processed/test_processed.csv")

test_unprocessed = pd.read_csv("../data/interim/test_unprocessed.csv")



X_test = test_final.drop(columns=["y"])
y_test = test_final["y"]

In [33]:
models_dir = r"C:\Users\DELL\OneDrive\Data Science\Bank_Telemarketing_Prediction_EDSB25_1\models"

# Load only LogisticRegression
logreg_model = load_model("LogisticRegression", models_dir)

# Use it directly
y_pred = logreg_model.predict(X_test)


✅ Loaded LogisticRegression from C:\Users\DELL\OneDrive\Data Science\Bank_Telemarketing_Prediction_EDSB25_1\models\LogisticRegression.pkl


In [34]:
y_pred

array([0, 1, 1, ..., 1, 0, 0], shape=(7743,))

Error Analysis

In [35]:
y_pred =logreg_model.predict(X_test)
y_proba = logreg_model.predict_proba(X_test)[:, 1]


In [36]:
errors = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred,
    "y_proba": y_proba
}, index=X_test.index)

false_positives = errors[(errors["y_true"] == 0) & (errors["y_pred"] == 1)]
false_negatives = errors[(errors["y_true"] == 1) & (errors["y_pred"] == 0)]

In [37]:
false_negatives

,y_true,y_pred,y_proba
4,1,0,0.322169
12,1,0,0.494657
27,1,0,0.481438
49,1,0,0.446911
66,1,0,0.436905
...,...,...,...
7726,1,0,0.275735
7727,1,0,0.445770
7736,1,0,0.385876
7738,1,0,0.417639


In [38]:
fn_analysis = test_unprocessed.loc[false_negatives.index]

In [39]:
fn_analysis

,job,marital,education,default,housing,loan,contact,month,day_of_week,age,pdays,emp_var_rate,cons_conf_idx,age_group,housing_loan_interaction
4,technician,married,university.degree,unknown,unknown,unknown,cellular,may,tue,43,999,-1.8,-46.2,adult,unknown_unknown
12,blue-collar,married,basic.4y,no,no,no,cellular,may,tue,28,999,-1.8,-46.2,adult,no_no
27,admin.,divorced,university.degree,no,yes,no,cellular,may,tue,37,999,-1.8,-46.2,adult,yes_no
49,blue-collar,married,basic.4y,no,yes,no,cellular,may,tue,43,999,-1.8,-46.2,adult,yes_no
66,management,married,basic.9y,no,yes,no,cellular,may,tue,48,999,-1.8,-46.2,adult,yes_no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7726,student,married,professional.course,no,yes,no,telephone,nov,thu,33,999,-1.1,-50.8,adult,yes_no
7727,admin.,single,university.degree,no,yes,no,cellular,nov,thu,31,999,-1.1,-50.8,adult,yes_no
7736,admin.,married,university.degree,no,yes,no,cellular,nov,fri,37,999,-1.1,-50.8,adult,yes_no
7738,retired,married,professional.course,no,yes,no,cellular,nov,fri,73,999,-1.1,-50.8,senior,yes_no


In [40]:
fn_analysis["education"].value_counts(normalize=True)

education
high.school            0.244048
university.degree      0.220238
professional.course    0.184524
basic.9y               0.154762
basic.4y               0.107143
basic.6y               0.047619
unknown                0.041667
Name: proportion, dtype: float64

In [46]:
# Build dataset
error_df = test_unprocessed.copy()
error_df["target"] = y_test
error_df["predicted"] = y_pred
error_df["predicted_proba"] = y_proba
error_df["match"] = (y_test == y_pred).astype(int)
error_df["diff"] = y_test - y_proba

# Example: group by a categorical feature
feature = "job"
grouped = error_df.groupby(feature).agg(
    match_rate=("match", "mean"),
    mean_diff=("diff", "mean")
)



In [48]:
grouped.sort_values(by="mean_diff", ascending=False)

,match_rate,mean_diff
job,,
housemaid,0.588571,-0.310381
unemployed,0.527132,-0.313598
services,0.497400,-0.372930
technician,0.483438,-0.384712
admin.,0.443409,-0.397703
entrepreneur,0.497143,-0.398336
self-employed,0.520161,-0.405767
management,0.450397,-0.409061
blue-collar,0.439580,-0.412687
